# Predicción de la posición habitual de un jugador de fútbol
## 04 · Base de datos

Proyecto final de Sistemas Inteligentes · UNAM · Matemáticas Aplicadas y Computación

## Objetivo

Guardar los datos del proyecto en una **base de datos**: el paso de la estructura que pidió la maestra para almacenar la información. Una base de datos es un archivo donde la información se guarda ordenada en **tablas** (como hojas de cálculo, pero conectadas entre sí y con reglas que evitan errores) y se consulta con un lenguaje llamado **SQL**.

Usamos **SQLite**: toda la base queda en un solo archivo (`Base_Datos_Futbol.db`) y no necesita instalar ningún servidor. Para llevarla a la nube, cada tabla se exporta también como CSV.

## Qué contiene

| Tabla | Qué guarda |
|---|---|
| `participantes` | Una fila por persona: edad, experiencia, nivel, posición, las 12 habilidades y las banderas de calidad |
| `respuestas_originales` | El texto original de cada respuesta (anónimo), ligado a `participantes` |
| `diccionario_variables` | A qué pregunta del formulario corresponde cada columna |
| `bitacora_limpieza` | Cada paso de la limpieza de datos |
| `kpis`, `perfil_habilidades`, `correlaciones_habilidades` | Resultados del análisis exploratorio |
| `resultados_modelos`, `comparacion_variables`, `comparacion_pca` | Comparación de los modelos |
| `resultados_por_posicion`, `matriz_confusion`, `importancia_variables` | Detalle del modelo elegido |
| `calibracion`, `sensibilidad` | ¿Son confiables las probabilidades? ¿Cambia algo si se quitan casos dudosos? |
| `resultados_kmeans`, `cruce_grupos_posicion` | Exploración no supervisada |
| `resumen_modelado` | Cifras clave del modelado |

Además hay **vistas** (consultas guardadas con nombre) que usan el tablero y la aplicación.

**Privacidad:** la base **no** tiene nombres, correos ni marcas de tiempo; solo códigos anónimos (`P001`...).

## Archivos que necesita

Los generaron los cuadernos 01, 02 y 03: `Datos_Limpios.csv`, `Respuestas_Originales_Anonimas.csv`, `Bitacora_Limpieza.csv`, `KPIs_EDA.csv`, `Perfil_Habilidades_por_Posicion.csv`, `Correlaciones_Habilidades.csv` y `Resultados_Modelado.zip`. Si no están en Colab, la celda de carga abre una ventana donde puedes elegirlos **todos juntos**.

### Preparar el entorno
*Qué hace:* importa las herramientas (`sqlite3` ya viene con Python).

In [1]:
import os
import re
import shutil
import sqlite3

import pandas as pd

try:
    from google.colab import files  # solo existe dentro de Google Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

try:
    from IPython.display import display
except ImportError:
    display = print

print("pandas", pd.__version__, "| SQLite", sqlite3.sqlite_version, "| Colab:", EN_COLAB)

pandas 2.2.3 | SQLite 3.45.1 | Colab: True


### Cargar los archivos
*Qué hace:* revisa que estén todos los archivos (si faltan, en Colab te pide subirlos), descomprime `Resultados_Modelado.zip` y lee cada tabla.

In [2]:
ARCHIVOS = [
    "Datos_Limpios.csv", "Respuestas_Originales_Anonimas.csv", "Bitacora_Limpieza.csv",
    "KPIs_EDA.csv", "Perfil_Habilidades_por_Posicion.csv", "Correlaciones_Habilidades.csv",
]
CARPETA = "resultados_modelado"
RESULTADOS = [
    "Resultados_Modelos.csv", "Comparacion_Variables.csv", "Comparacion_PCA.csv", "Resultados_Por_Posicion.csv",
    "Matriz_Confusion.csv", "Importancia_Variables.csv", "Calibracion.csv", "Sensibilidad.csv",
    "Resultados_KMeans.csv", "Cruce_Grupos_Posicion.csv", "Resumen_Modelado.csv",
]


def faltantes():
    lista = [a for a in ARCHIVOS if not os.path.exists(a)]
    if not all(os.path.exists(os.path.join(CARPETA, r)) for r in RESULTADOS) and not os.path.exists("Resultados_Modelado.zip"):
        lista.append("Resultados_Modelado.zip")
    return lista


falta = faltantes()
if falta and EN_COLAB:
    print("Sube estos archivos (puedes elegirlos todos juntos):", falta)
    files.upload()
    falta = faltantes()
if falta:
    raise FileNotFoundError(f"Faltan estos archivos: {falta}. Súbelos con su nombre exacto.")

if not all(os.path.exists(os.path.join(CARPETA, r)) for r in RESULTADOS):
    shutil.unpack_archive("Resultados_Modelado.zip", CARPETA)


def leer(ruta):
    return pd.read_csv(ruta, encoding="utf-8-sig")


participantes = leer("Datos_Limpios.csv")
originales = leer("Respuestas_Originales_Anonimas.csv")
bitacora = leer("Bitacora_Limpieza.csv").rename(columns={"n": "numero"})
kpis = leer("KPIs_EDA.csv")
perfil = leer("Perfil_Habilidades_por_Posicion.csv")
correlaciones = leer("Correlaciones_Habilidades.csv")
modelado = {r: leer(os.path.join(CARPETA, r)) for r in RESULTADOS}

print("Participantes:", len(participantes), "| Respuestas originales:", len(originales), "| Pasos de bitácora:", len(bitacora))

Participantes: 272 | Respuestas originales: 272 | Pasos de bitácora: 19


### Diccionario de variables
*Qué hace:* crea una tabla que explica qué es cada columna de `participantes` y a qué pregunta del formulario corresponde.

In [3]:
HABILIDADES = [
    ("velocidad", 7, "Velocidad"), ("resistencia", 8, "Resistencia física"), ("fuerza", 9, "Fuerza física"),
    ("control_balon", 10, "Control del balón"), ("regate", 11, "Capacidad de regate"), ("pase", 12, "Precisión de pase"),
    ("definicion", 13, "Capacidad de definición o remate"), ("defensa", 14, "Capacidad defensiva"),
    ("juego_aereo", 15, "Juego aéreo"), ("vision_juego", 16, "Visión de juego"), ("reflejos", 17, "Reflejos"),
    ("posicionamiento", 18, "Posicionamiento táctico"),
]
filas = [
    ("id_persona", None, "Código anónimo de la persona (P001...)", "texto", "P001 a P272"),
    ("edad", 2, "Edad en años", "entero", "13 a 90"),
    ("tiempo_juego", 3, "Cuánto tiempo lleva jugando", "texto ordenado", "Menos de 1 año ... Más de 10 años"),
    ("tiempo_codigo", 3, "Código numérico del tiempo jugando", "entero", "0 a 4"),
    ("horas_semana", 4, "Horas por semana que juega o entrena", "texto ordenado", "Actualmente, ninguna ... Más de 12 horas"),
    ("horas_codigo", 4, "Código numérico de las horas por semana", "entero", "0 a 5"),
    ("nivel", 5, "Nivel en que juega o ha jugado principalmente", "texto", "Escolar ... Profesional"),
    ("posicion", 6, "Posición habitual (VARIABLE OBJETIVO)", "texto", "7 posiciones"),
]
filas += [(c, n, f"Autoevaluación de: {t}", "entero (puede faltar)", "1 muy bajo ... 5 muy alto; vacío = 'No sé'") for c, n, t in HABILIDADES]
filas += [
    ("n_no_se", None, "Cantidad de habilidades que la persona no supo evaluar", "entero", "0 a 12"),
    ("duplicado_posible", None, "1 si sus 18 respuestas son idénticas a las de otra persona", "bandera 0/1", "0 o 1"),
    ("duplicado_repetida", None, "1 si es la segunda o posterior respuesta idéntica", "bandera 0/1", "0 o 1"),
    ("menor_18", None, "1 si tiene menos de 18 años", "bandera 0/1", "0 o 1"),
    ("edad_atipica", None, "1 si la edad está fuera del rango intercuartílico", "bandera 0/1", "0 o 1"),
    ("linea_recta", None, "1 si calificó igual todas las habilidades", "bandera 0/1", "0 o 1"),
    ("incoherencia_experiencia", None, "1 si la edad no cuadra con el tiempo jugando", "bandera 0/1", "0 o 1"),
]
diccionario = pd.DataFrame(filas, columns=["variable", "pregunta", "descripcion", "tipo", "valores"])
diccionario["pregunta"] = diccionario["pregunta"].astype("Int64")

assert set(diccionario["variable"]) == set(participantes.columns), "El diccionario no coincide con las columnas de participantes."
print("Variables descritas:", len(diccionario))

Variables descritas: 27


## Crear la base de datos
*Qué hace:* crea el archivo `Base_Datos_Futbol.db` y sus tablas. Cada tabla se crea con su **estructura definida** (tipos de dato, llave primaria y reglas), no solo pegando datos:

- `id_persona` es **llave primaria**: no puede repetirse.
- `posicion` solo acepta las **siete posiciones válidas** (`CHECK`).
- Cada habilidad solo acepta valores de **1 a 5** o vacío (`CHECK`).
- `respuestas_originales` está **ligada** a `participantes` por `id_persona` (llave foránea): no puede existir una respuesta sin su persona.

In [4]:
RUTA_DB = "Base_Datos_Futbol.db"
if os.path.exists(RUTA_DB):
    os.remove(RUTA_DB)  # se crea desde cero para que el resultado sea siempre el mismo

con = sqlite3.connect(RUTA_DB)
con.execute("PRAGMA foreign_keys = ON")


def tipo_sql(serie):
    if pd.api.types.is_integer_dtype(serie):
        return "INTEGER"
    if pd.api.types.is_float_dtype(serie):
        return "REAL"
    return "TEXT"


def crear_tabla(nombre, tabla, clave=None, reglas_extra=()):
    columnas = []
    for c in tabla.columns:
        definicion = f'"{c}" {tipo_sql(tabla[c])}'
        if c == clave:
            definicion += " PRIMARY KEY"
        columnas.append(definicion)
    columnas += list(reglas_extra)
    con.execute(f'CREATE TABLE "{nombre}" ({", ".join(columnas)})')
    tabla.to_sql(nombre, con, if_exists="append", index=False)
    print(f"Tabla {nombre}: {len(tabla)} filas, {len(tabla.columns)} columnas")


POSICIONES = ["Portero", "Defensa central", "Lateral", "Mediocampista defensivo",
              "Mediocampista ofensivo", "Extremo", "Delantero"]
regla_posicion = "CHECK (posicion IN (" + ", ".join(f"'{p}'" for p in POSICIONES) + "))"
reglas_habilidades = [f'CHECK ("{c}" IS NULL OR "{c}" BETWEEN 1 AND 5)' for c, _, _ in HABILIDADES]
reglas_banderas = [f'CHECK ("{c}" IN (0, 1))' for c in
                   ["duplicado_posible", "duplicado_repetida", "menor_18", "edad_atipica", "linea_recta", "incoherencia_experiencia"]]

crear_tabla("participantes", participantes, clave="id_persona",
            reglas_extra=[regla_posicion, 'CHECK (edad BETWEEN 5 AND 100)'] + reglas_habilidades + reglas_banderas)
crear_tabla("respuestas_originales", originales, clave="id_persona",
            reglas_extra=[regla_posicion, "FOREIGN KEY (id_persona) REFERENCES participantes (id_persona)"])
crear_tabla("diccionario_variables", diccionario)
crear_tabla("bitacora_limpieza", bitacora)
crear_tabla("kpis", kpis)
crear_tabla("perfil_habilidades", perfil)
crear_tabla("correlaciones_habilidades", correlaciones)

TABLAS_MODELADO = {
    "resultados_modelos": "Resultados_Modelos.csv",
    "comparacion_variables": "Comparacion_Variables.csv",
    "comparacion_pca": "Comparacion_PCA.csv",
    "resultados_por_posicion": "Resultados_Por_Posicion.csv",
    "matriz_confusion": "Matriz_Confusion.csv",
    "importancia_variables": "Importancia_Variables.csv",
    "calibracion": "Calibracion.csv",
    "sensibilidad": "Sensibilidad.csv",
    "resultados_kmeans": "Resultados_KMeans.csv",
    "cruce_grupos_posicion": "Cruce_Grupos_Posicion.csv",
    "resumen_modelado": "Resumen_Modelado.csv",
}
for tabla, archivo in TABLAS_MODELADO.items():
    crear_tabla(tabla, modelado[archivo])
con.commit()

Tabla participantes: 272 filas, 27 columnas
Tabla respuestas_originales: 272 filas, 19 columnas
Tabla diccionario_variables: 27 filas, 5 columnas
Tabla bitacora_limpieza: 19 filas, 4 columnas
Tabla kpis: 16 filas, 3 columnas
Tabla perfil_habilidades: 7 filas, 13 columnas
Tabla correlaciones_habilidades: 12 filas, 13 columnas
Tabla resultados_modelos: 9 filas, 8 columnas
Tabla comparacion_variables: 8 filas, 4 columnas
Tabla comparacion_pca: 3 filas, 4 columnas
Tabla resultados_por_posicion: 7 filas, 5 columnas
Tabla matriz_confusion: 7 filas, 8 columnas
Tabla importancia_variables: 12 filas, 3 columnas
Tabla calibracion: 6 filas, 4 columnas
Tabla sensibilidad: 3 filas, 5 columnas
Tabla resultados_kmeans: 9 filas, 5 columnas
Tabla cruce_grupos_posicion: 2 filas, 8 columnas
Tabla resumen_modelado: 23 filas, 2 columnas


### Crear las vistas
*Qué hace:* crea **vistas**, que son consultas guardadas con nombre. Reúnen resúmenes que usan el tablero y la aplicación, calculados **directamente con SQL** a partir de `participantes`.

In [5]:
CORTOS = {
    "velocidad": "Velocidad", "resistencia": "Resistencia", "fuerza": "Fuerza", "control_balon": "Control del balón",
    "regate": "Regate", "pase": "Pase", "definicion": "Definición", "defensa": "Defensa",
    "juego_aereo": "Juego aéreo", "vision_juego": "Visión de juego", "reflejos": "Reflejos",
    "posicionamiento": "Posicionamiento",
}
promedios = ", ".join(f'ROUND(AVG("{c}"), 3) AS "{CORTOS[c]}"' for c, _, _ in HABILIDADES)
vistas = {
    "v_resumen_posiciones": (
        "SELECT posicion, COUNT(*) AS personas, "
        "ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM participantes), 1) AS porcentaje "
        "FROM participantes GROUP BY posicion ORDER BY personas DESC"
    ),
    "v_perfil_habilidades_sql": f"SELECT posicion, COUNT(*) AS personas, {promedios} FROM participantes GROUP BY posicion",
    "v_nivel_posicion": "SELECT nivel, posicion, COUNT(*) AS personas FROM participantes GROUP BY nivel, posicion",
    "v_edad_por_posicion": (
        "SELECT posicion, COUNT(*) AS personas, ROUND(AVG(edad), 1) AS edad_promedio, "
        "MIN(edad) AS edad_minima, MAX(edad) AS edad_maxima FROM participantes GROUP BY posicion"
    ),
    "v_casos_marcados": (
        "SELECT id_persona, edad, nivel, posicion, duplicado_posible, duplicado_repetida, menor_18, "
        "edad_atipica, linea_recta, incoherencia_experiencia FROM participantes "
        "WHERE duplicado_posible + menor_18 + edad_atipica + linea_recta + incoherencia_experiencia > 0"
    ),
    "v_modelos_ordenados": (
        "SELECT modelo, ROUND(aciertos, 3) AS aciertos, ROUND(aciertos_balanceados, 3) AS aciertos_balanceados, "
        "ROUND(f1_macro, 3) AS f1_macro, ROUND(top2, 3) AS top2 FROM resultados_modelos ORDER BY f1_macro DESC"
    ),
}
for nombre, consulta in vistas.items():
    con.execute(f'CREATE VIEW "{nombre}" AS {consulta}')
con.commit()
print("Vistas creadas:", list(vistas))

Vistas creadas: ['v_resumen_posiciones', 'v_perfil_habilidades_sql', 'v_nivel_posicion', 'v_edad_por_posicion', 'v_casos_marcados', 'v_modelos_ordenados']


## Comprobar que la base quedó bien
*Qué hace:* verifica que (1) cada tabla tenga las mismas filas que el archivo de origen, (2) la base no esté dañada, (3) las llaves foráneas sean correctas, (4) los resúmenes calculados con SQL **coincidan** con los del análisis en Python y (5) **no haya datos personales** (ni columnas ni valores que parezcan correos, marcas de tiempo o teléfonos).

In [6]:
origen = {
    "participantes": participantes, "respuestas_originales": originales, "diccionario_variables": diccionario,
    "bitacora_limpieza": bitacora, "kpis": kpis, "perfil_habilidades": perfil,
    "correlaciones_habilidades": correlaciones,
}
origen.update({tabla: modelado[archivo] for tabla, archivo in TABLAS_MODELADO.items()})

for tabla, datos in origen.items():
    en_base = con.execute(f'SELECT COUNT(*) FROM "{tabla}"').fetchone()[0]
    assert en_base == len(datos), f"{tabla}: {en_base} filas en la base y {len(datos)} en el archivo"
print("1) Las filas de cada tabla coinciden con los archivos de origen: OK")

assert con.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
print("2) Integridad de la base: OK")

assert con.execute("PRAGMA foreign_key_check").fetchall() == []
print("3) Llaves foráneas: OK")

sql = pd.read_sql_query("SELECT * FROM v_perfil_habilidades_sql", con).set_index("posicion")
referencia = perfil.set_index("posicion")
sql = sql.loc[referencia.index, referencia.columns]
assert ((sql - referencia).abs().max().max()) < 0.005, "El perfil calculado con SQL no coincide con el de Python."
total_vista = con.execute("SELECT SUM(personas) FROM v_resumen_posiciones").fetchone()[0]
assert total_vista == len(participantes)
print("4) Los resúmenes en SQL coinciden con los del análisis en Python: OK")

patron_columna = re.compile(r"correo|e-?mail|cuenta|marca|fecha|nombre|tel[eé]fono", re.IGNORECASE)
patron_dato = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+|\d{4}/\d{2}/\d{2}|\b\d{10}\b")
for tabla in origen:
    info = con.execute(f'PRAGMA table_info("{tabla}")').fetchall()
    assert not any(patron_columna.search(fila[1]) for fila in info), f"columna sospechosa en {tabla}"
for tabla in ("participantes", "respuestas_originales"):
    datos = origen[tabla]
    for columna in datos.columns:
        textos = [v for v in datos[columna].dropna() if isinstance(v, str)]
        assert not any(patron_dato.search(v) for v in textos), f"dato personal en {tabla}.{columna}"
print("5) Sin correos, fechas, teléfonos ni nombres: OK")

1) Las filas de cada tabla coinciden con los archivos de origen: OK
2) Integridad de la base: OK
3) Llaves foráneas: OK
4) Los resúmenes en SQL coinciden con los del análisis en Python: OK
5) Sin correos, fechas, teléfonos ni nombres: OK


## Consultas de ejemplo (SQL)
*Qué hace:* ejecuta cinco consultas para mostrar cómo se usa la base. Se guardan también en `consultas_ejemplo.sql`.

In [7]:
CONSULTAS = {
    "¿Cuántas personas hay en cada posición?":
        "SELECT * FROM v_resumen_posiciones",
    "¿Cuáles son las habilidades más altas de cada posición? (promedios de 1 a 5)":
        'SELECT posicion, "Regate", "Definición", "Defensa", "Reflejos", "Juego aéreo" FROM v_perfil_habilidades_sql',
    "¿Qué modelo tuvo mejores resultados?":
        "SELECT * FROM v_modelos_ordenados",
    "¿Qué extremos se calificaron con regate 5? (une dos tablas: valor limpio y respuesta original)":
        "SELECT p.id_persona, p.edad, p.regate, o.regate AS regate_original FROM participantes p "
        "JOIN respuestas_originales o ON o.id_persona = p.id_persona "
        "WHERE p.posicion = 'Extremo' AND p.regate = 5 ORDER BY p.id_persona LIMIT 8",
    "¿Cuántos casos marcados hay y de qué tipo?":
        "SELECT SUM(duplicado_posible) AS duplicados_posibles, SUM(duplicado_repetida) AS duplicados_repetidos, "
        "SUM(menor_18) AS menores_18, SUM(edad_atipica) AS edades_atipicas, SUM(linea_recta) AS linea_recta, "
        "SUM(incoherencia_experiencia) AS incoherencias FROM participantes",
}

with open("consultas_ejemplo.sql", "w", encoding="utf-8") as archivo:
    for pregunta, consulta in CONSULTAS.items():
        archivo.write(f"-- {pregunta}\n{consulta};\n\n")

for pregunta, consulta in CONSULTAS.items():
    print("\n" + pregunta)
    display(pd.read_sql_query(consulta, con))


¿Cuántas personas hay en cada posición?


,posicion,personas,porcentaje
0,Defensa central,53,19.5
1,Delantero,43,15.8
2,Lateral,41,15.1
3,Mediocampista defensivo,39,14.3
4,Mediocampista ofensivo,37,13.6
5,Portero,32,11.8
6,Extremo,27,9.9



¿Cuáles son las habilidades más altas de cada posición? (promedios de 1 a 5)


,posicion,Regate,Definición,Defensa,Reflejos,Juego aéreo
0,Defensa central,2.077,2.192,3.942,2.942,3.885
1,Delantero,3.286,4.023,2.233,3.512,3.116
2,Extremo,4.519,3.630,2.556,3.815,2.296
3,Lateral,2.951,2.366,3.634,3.171,2.707
4,Mediocampista defensivo,2.816,2.692,3.923,3.385,3.282
5,Mediocampista ofensivo,3.622,3.270,2.784,3.324,2.432
6,Portero,1.563,1.656,3.125,4.406,3.313



¿Qué modelo tuvo mejores resultados?


,modelo,aciertos,aciertos_balanceados,f1_macro,top2
0,Bosque aleatorio,0.686,0.689,0.693,0.800
1,Regresión logística + PCA,0.688,0.693,0.692,0.829
2,Regresión logística,0.684,0.689,0.688,0.827
3,SVM,0.664,0.671,0.674,0.804
4,SVM + PCA,0.652,0.658,0.661,0.822
5,k vecinos,0.640,0.643,0.648,0.748
6,k vecinos + PCA,0.645,0.648,0.647,0.778
7,Árbol de decisión,0.607,0.608,0.608,0.695
8,Base (siempre la posición más común),0.195,0.143,0.047,0.353



¿Qué extremos se calificaron con regate 5? (une dos tablas: valor limpio y respuesta original)


,id_persona,edad,regate,regate_original
0,P089,39,5.0,5 — Muy alto/a
1,P090,31,5.0,5 — Muy alto/a
2,P097,47,5.0,5 — Muy alto/a
3,P105,62,5.0,5 — Muy alto/a
4,P130,17,5.0,5 — Muy alto/a
5,P136,20,5.0,5 — Muy alto/a
6,P154,36,5.0,5 — Muy alto/a
7,P169,17,5.0,5 — Muy alto/a



¿Cuántos casos marcados hay y de qué tipo?


,duplicados_posibles,duplicados_repetidos,menores_18,edades_atipicas,linea_recta,incoherencias
0,14,7,33,1,4,0


## Exportar para la nube
*Qué hace:* guarda cada tabla como archivo **CSV** en un `.zip`. Con esos CSV se puede subir la base a un servicio en la nube (por ejemplo BigQuery o Google Sheets) y conectarla a un tablero. Después cierra la base y, en Colab, descarga los archivos.

In [8]:
os.makedirs("exportes_csv", exist_ok=True)
tablas = [f[0] for f in con.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")]
for tabla in tablas:
    pd.read_sql_query(f'SELECT * FROM "{tabla}"', con).to_csv(f"exportes_csv/{tabla}.csv", index=False, encoding="utf-8")

shutil.make_archive("exportes_csv", "zip", "exportes_csv")
n_tablas = len(tablas)
n_vistas = con.execute("SELECT COUNT(*) FROM sqlite_master WHERE type = 'view'").fetchone()[0]
con.close()

print(f"Tablas exportadas ({n_tablas}):", tablas)
print("Archivos creados: Base_Datos_Futbol.db, consultas_ejemplo.sql, exportes_csv.zip")
if EN_COLAB:
    for archivo in ("Base_Datos_Futbol.db", "consultas_ejemplo.sql", "exportes_csv.zip"):
        files.download(archivo)

Tablas exportadas (18): ['bitacora_limpieza', 'calibracion', 'comparacion_pca', 'comparacion_variables', 'correlaciones_habilidades', 'cruce_grupos_posicion', 'diccionario_variables', 'importancia_variables', 'kpis', 'matriz_confusion', 'participantes', 'perfil_habilidades', 'respuestas_originales', 'resultados_kmeans', 'resultados_modelos', 'resultados_por_posicion', 'resumen_modelado', 'sensibilidad']
Archivos creados: Base_Datos_Futbol.db, consultas_ejemplo.sql, exportes_csv.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Resumen de esta ejecución
*Qué hace:* imprime las cifras reales de esta ejecución.

In [9]:
print("RESUMEN DE LA EJECUCIÓN")
print("-" * 40)
print(f"Tablas: {n_tablas} | Vistas: {n_vistas}")
print(f"Personas en la base: {len(participantes)}")
print(f"Respuestas originales ligadas: {len(originales)}")
print("Comprobaciones: filas, integridad, llaves foráneas, coherencia con Python y privacidad: OK")

RESUMEN DE LA EJECUCIÓN
----------------------------------------
Tablas: 18 | Vistas: 6
Personas en la base: 272
Respuestas originales ligadas: 272
Comprobaciones: filas, integridad, llaves foráneas, coherencia con Python y privacidad: OK


## Conclusiones

- La base `Base_Datos_Futbol.db` guarda los datos limpios y anónimos, las respuestas originales, la bitácora de limpieza, los resultados del análisis exploratorio y los del modelado.
- Está **bien estructurada**: llaves primarias, una llave foránea entre `respuestas_originales` y `participantes`, y reglas que solo permiten posiciones válidas, habilidades de 1 a 5 y banderas 0/1.
- Se comprobó su integridad, que los resúmenes calculados con SQL **coinciden** con los del análisis en Python y que **no contiene datos personales**.
- Es una base **local** (un archivo). Para llevarla a la nube se exportan las tablas a CSV.

**Siguiente paso:** la aplicación web y el tablero analítico.